# NISAR ISCE3 PGE

Papermill-driven notebook PGE. Runs under the **`isce3_src`** kernel so that `nisar`/`isce3` (and the `stage_dem` subprocess spawned by the localizer) resolve.

Steps:
1. Write netrc credentials, put `nisar_products` on the path, point at the track/frame DB.
2. Localize the S3 runconfig (download inputs, stage DEM) to local paths.
3. Dispatch the NISAR SAS workflow for the runconfig product type.
4. Stage out every deliverable HDF5: rename to its full NISAR granule name (read from HDF5 metadata) and move each into a sibling directory named for the granule, alongside a copy of the runconfig used. A combined InSAR runconfig (e.g. `product_type: RIFG_RUNW_GUNW`) yields multiple products.

In [ ]:
# Papermill parameters. Override at run time with `papermill -p ...`.
runconfig_s3 = ""          # inline runconfig YAML text (containing s3:// links), or a local path
netrc_content = ""         # netrc credentials text (machine/login/password)
output_dir = "output"      # SAS product output directory
scratch_dir = "scratch"    # SAS scratch directory
localized_runconfig = "runconfig_localized.yaml"  # written localized runconfig
mozart_pvt_ip = ""         # Mozart private IP; worker mints DAAC creds via https://<ip>:8888/api/v0.1/daac/s3credentials (bypasses the httpd basic-auth proxy)

# hysds specifications
_time_limit = 172800
_soft_time_limit = 172800
_disk_usage = '200GB'
_submission_type = 'iteration'
_label = 'ISCE3 PGE'

In [ ]:
import os
import re
import sys
import json
import stat
import shutil
import subprocess
from pathlib import Path

import yaml
import h5py

# Fail loud if we are not running under the isce3_src kernel: nisar must import
# in-process here, and the localizer's `stage_dem` subprocess uses sys.executable.
import nisar  # noqa: F401
print(f"interpreter: {sys.executable}")

# Make nisar_products importable. It is pip-installed into isce3_src in the image;
# fall back to the source checkout for local/dev runs.
try:
    import nisar_products  # noqa: F401
except ImportError:
    src = os.environ.get("NISAR_ONDEMAND_SRC", "/home/jovyan/ondemand-resources/src")
    if src not in sys.path:
        sys.path.insert(0, src)
    import nisar_products  # noqa: F401

from nisar_products import configure
from nisar_products.runconfig_localizer import localize_runconfig

# Track/frame GeoPackage shipped with this repo. Constant (not a papermill param):
# it is a fixed reference DB used only by the INSAR DEM-bbox fallback.
TRACKFRAME_DB = "/home/jovyan/nisar-isce3-pge/data/NISAR_TrackFrame_L_20260618.gpkg"

# Write netrc so earthaccess can fetch temporary DAAC S3 credentials.
if netrc_content.strip():
    netrc_path = Path.home() / ".netrc"
    netrc_path.write_text(netrc_content if netrc_content.endswith("\n") else netrc_content + "\n")
    netrc_path.chmod(stat.S_IRUSR | stat.S_IWUSR)  # 0600, required by netrc consumers
    print(f"wrote {netrc_path}")

# Point the track/frame lookup at this repo's GeoPackage.
configure(trackframe_db=TRACKFRAME_DB)
print(f"trackframe_db: {TRACKFRAME_DB}")

In [ ]:
# Fetch short-term DAAC S3 credentials from Mozart, for workers with NO public
# internet egress. The worker cannot reach urs.earthdata.nasa.gov to mint DAAC
# creds via earthaccess, but it CAN reach Mozart on the internal subnet. We call
# the Mozart gunicorn API DIRECTLY on port 8888 -- https://<pvt_ip>:8888/api/v0.1/
# daac/s3credentials -- which bypasses the Apache httpd proxy that fronts the
# public `/mozart/...` path and enforces basic-auth (that proxy 401s the worker).
# The endpoint takes the netrc text, logs in to Earthdata server-side, and returns
# short-term accessKeyId/secretAccessKey/sessionToken. We write those into the
# `[daac]` AWS profile, which nisar_products.s3_search prefers for DAAC buckets
# (get_s3_client_for_bucket -> NISAR_DAAC_PROFILE, default "daac").
#
# BEST EFFORT: this must never crash the job. On any failure it logs a clear,
# non-silent error and continues -- localization then falls back to the profile /
# earthaccess / default-chain path as before.
def _fetch_daac_creds_via_mozart(mozart_pvt_ip, netrc_text):
    """POST netrc to Mozart's DAAC cred endpoint (direct on :8888); write creds to
    the [daac] AWS profile. Returns True on success, False otherwise. Never raises."""
    import configparser  # stdlib

    try:
        import requests
    except Exception as exc:  # noqa: BLE001
        print(f"[daac-creds] ERROR: `requests` unavailable, cannot call Mozart: {exc}",
              flush=True)
        return False

    # Hit the gunicorn API directly on :8888 (path is /api/v0.1/..., NOT
    # /mozart/api/v0.1/...), bypassing the httpd basic-auth proxy.
    ip = mozart_pvt_ip.strip().rstrip("/")
    url = f"https://{ip}:8888/api/v0.1/daac/s3credentials"

    try:
        # verify=False: the internal Mozart host uses a self-signed/instance cert,
        # matching otello's own default on the submit side. urllib3 will warn once.
        resp = requests.post(url, data={"netrc": netrc_text}, timeout=60, verify=False)
    except Exception as exc:  # noqa: BLE001 -- network/TLS/DNS failure
        print(f"[daac-creds] ERROR: request to {url} failed: {exc}", flush=True)
        return False

    if resp.status_code != 200:
        # Body may carry a useful message; it never contains the netrc or creds.
        print(f"[daac-creds] ERROR: Mozart returned HTTP {resp.status_code}: "
              f"{resp.text[:500]}", flush=True)
        return False

    try:
        body = resp.json()
        creds = body["credentials"]
        key_id = creds["accessKeyId"]
        secret = creds["secretAccessKey"]
        token = creds["sessionToken"]
    except Exception as exc:  # noqa: BLE001 -- malformed payload
        print(f"[daac-creds] ERROR: could not parse Mozart credential payload: {exc}",
              flush=True)
        return False

    if not (key_id and secret and token):
        print("[daac-creds] ERROR: Mozart returned an incomplete credential set",
              flush=True)
        return False

    # Write/merge the [daac] profile into ~/.aws/credentials (the file the worker
    # mounts). Preserve any other profiles already present.
    try:
        creds_path = os.path.expanduser(
            os.environ.get("AWS_SHARED_CREDENTIALS_FILE", "~/.aws/credentials"))
        os.makedirs(os.path.dirname(creds_path), exist_ok=True)
        cp = configparser.ConfigParser()
        if os.path.exists(creds_path):
            cp.read(creds_path)
        if not cp.has_section("daac"):
            cp.add_section("daac")
        cp["daac"]["aws_access_key_id"] = key_id
        cp["daac"]["aws_secret_access_key"] = secret
        cp["daac"]["aws_session_token"] = token
        with open(creds_path, "w") as f:
            cp.write(f)
        os.chmod(creds_path, stat.S_IRUSR | stat.S_IWUSR)  # 0600
    except Exception as exc:  # noqa: BLE001
        print(f"[daac-creds] ERROR: obtained creds but failed to write [daac] "
              f"profile: {exc}", flush=True)
        return False

    exp = creds.get("expiration")
    print(f"[daac-creds] wrote short-term [daac] AWS profile from Mozart "
          f"(expires: {exp})", flush=True)
    return True


if mozart_pvt_ip.strip() and netrc_content.strip():
    print(f"[daac-creds] requesting DAAC S3 credentials from Mozart at "
          f"https://{mozart_pvt_ip.strip()}:8888", flush=True)
    _mozart_creds_ok = _fetch_daac_creds_via_mozart(mozart_pvt_ip, netrc_content)
    if not _mozart_creds_ok:
        print("[daac-creds] WARNING: could not obtain DAAC creds via Mozart; "
              "localization will fall back to earthaccess / default AWS creds "
              "(may 403 on DAAC buckets if the worker has no internet)", flush=True)
elif not mozart_pvt_ip.strip():
    print("[daac-creds] mozart_pvt_ip not set; skipping Mozart DAAC credential fetch",
          flush=True)

In [ ]:
# The runconfig is delivered inline as YAML text (a `destination: context` param,
# so HySDS stores the submitted value verbatim). Materialize it to a local file
# for the localizer, which reads its argument with a plain open(). A bare local
# path is still accepted for back-compat / local dev runs.
if runconfig_s3 and os.path.isfile(runconfig_s3):
    rc_input = runconfig_s3
    print(f"using runconfig path: {rc_input}")
else:
    if not runconfig_s3.strip():
        raise ValueError("runconfig_s3 parameter is empty")

    rc_input = "runconfig_input.yaml"
    Path(rc_input).write_text(runconfig_s3)
    print(f"wrote inline runconfig to {rc_input} ({len(runconfig_s3)} bytes)")

# Download every s3:// input in the runconfig, stage the DEM, rewrite paths to
# local absolute paths, and pin the output/scratch dirs. Returns the local YAML path.
local_rc = localize_runconfig(
    rc_input,
    localized_runconfig,
    output_dir=output_dir,
    scratch_dir=scratch_dir,
)
print(f"localized runconfig: {local_rc}")

In [ ]:
# Resolve workflow module + the list of deliverable HDF5 files from the localized runconfig.
with open(local_rc) as f:
    cfg = yaml.safe_load(f)
groups = cfg["runconfig"]["groups"]
product_type = groups["primary_executable"]["product_type"]
sas_output_file = groups["product_path_group"]["sas_output_file"]

# product_type -> nisar.workflows module. The InSAR family is one umbrella module and its
# product_type may be a single code (RIFG) or an underscore-joined combo (RIFG_RUNW_GUNW).
SINGLE_MODULE = {"RSLC": "focus", "GSLC": "gslc", "GCOV": "gcov", "SME2": "sme2"}
INSAR_CODES = {"RIFG", "RUNW", "ROFF", "GUNW", "GOFF"}

if product_type in SINGLE_MODULE:
    module = SINGLE_MODULE[product_type]
elif set(product_type.split("_")) <= INSAR_CODES:
    module = "insar"
else:
    raise ValueError(f"unsupported product_type {product_type!r}")

# Enumerate the HDF5 files the workflow will emit. For InSAR, reuse the SAS's own mapping
# (h5_prep.get_products_and_paths) so combined types like RIFG_RUNW_GUNW yield every
# co-product (output/RIFG_product.h5, output/RUNW_product.h5, output/GUNW_product.h5).
# Intermediates written to the scratch dir are NOT deliverables.
if module == "insar":
    from nisar.workflows.h5_prep import get_products_and_paths
    _subprods, h5_paths = get_products_and_paths(groups)
    scratch_dir_abs = os.path.abspath(groups["product_path_group"]["scratch_path"])
    deliverables = [
        p for p in h5_paths.values()
        if os.path.abspath(os.path.dirname(p)) != scratch_dir_abs
    ]
else:
    deliverables = [sas_output_file]

print(f"product_type={product_type} module=nisar.workflows.{module}")
print("deliverables:", deliverables)

In [ ]:
# Run the SAS workflow in the isce3_src conda env so its activation scripts set
# PYTHONPATH/LD_LIBRARY_PATH for libisce3.
#
# We WANT `conda run -n isce3_src` to work (portable, also correct for local dev
# where the env prefix differs). `-n` resolves the NAME against conda's envs_dirs,
# which the Dockerfile now registers (conda config --append envs_dirs
# /opt/conda/envs). But envs_dirs is a runtime property (conda root + HOME +
# condarc), so verify it actually took here rather than assume: probe `-n` first,
# log the outcome, and fall back to the known absolute prefix if it did not.
ISCE3_ENV_NAME = "isce3_src"
ISCE3_ENV_PREFIX = "/opt/conda/envs/isce3_src"  # Dockerfile: --envs-dir/--env-name

# Cheap resolution check: does `conda run -n isce3_src` find the env at all?
probe = subprocess.run(
    ["conda", "run", "-n", ISCE3_ENV_NAME, "python", "-c", "import sys; print(sys.prefix)"],
    capture_output=True, text=True,
)
if probe.returncode == 0:
    conda_target = ["-n", ISCE3_ENV_NAME]
    print(f"[env] `-n {ISCE3_ENV_NAME}` RESOLVES -> {probe.stdout.strip()} "
          f"(name resolution fixed)", flush=True)
else:
    conda_target = ["-p", ISCE3_ENV_PREFIX]
    print(f"[env] `-n {ISCE3_ENV_NAME}` FAILED to resolve; falling back to "
          f"`-p {ISCE3_ENV_PREFIX}`. conda said:\n{probe.stderr.strip()}", flush=True)

# --live-stream forwards stdout/stderr in real time instead of buffering.
cmd = [
    "conda", "run", *conda_target, "--live-stream",
    "python", "-m", f"nisar.workflows.{module}", local_rc,
]
print("running:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)
print("SAS workflow completed")

In [ ]:
# Stage out each deliverable. The SAS writes the fully-substituted granule name into each
# product HDF5. For every deliverable: read its granuleId, create a sibling directory (next
# to output/) named after the granule, MOVE the HDF5 into it as <granuleId>.h5, drop a copy
# of the localized runconfig, and write the HySDS sidecars <granuleId>.dataset.json and
# <granuleId>.met.json so the product ingests with searchable metadata. Every sidecar field
# is read from the product HDF5's /science/LSAR/identification group -- nothing hardcoded.


def _h5_str(h5, path):
    """Return a scalar HDF5 dataset as a stripped str, or None if absent/empty."""
    try:
        val = h5[path][()]
    except (KeyError, TypeError):
        return None
    if isinstance(val, bytes):
        val = val.decode()
    val = str(val).strip()
    return val or None


def _h5_num(h5, path):
    """Return a scalar HDF5 dataset as an int/float, or None if absent."""
    try:
        val = h5[path][()]
    except (KeyError, TypeError):
        return None
    try:
        f = float(val)
    except (TypeError, ValueError):
        return None
    return int(f) if f.is_integer() else f


def _bounding_polygon_to_geojson(wkt):
    """Convert a NISAR ``boundingPolygon`` WKT string to GeoJSON polygon coordinates.

    The identification/boundingPolygon dataset is a WKT ``POLYGON ((lon lat, ...))``
    (vertices may carry a third Z value, which we drop). Returns a GeoJSON-style
    ``[[[lon, lat], ...]]`` ring list, or None if it cannot be parsed.
    """
    if not wkt:
        return None
    m = re.search(r"\(\(\s*(.*?)\s*\)\)", wkt, re.DOTALL)
    if not m:
        return None
    ring = []
    for vertex in m.group(1).split(","):
        nums = vertex.split()
        if len(nums) < 2:
            continue
        try:
            ring.append([float(nums[0]), float(nums[1])])
        except ValueError:
            return None
    return [ring] if ring else None


def _safe_write_json(path, payload):
    """Write ``payload`` as JSON to ``path``, never raising.

    Returns True on success. On any failure, attempts to write a minimal static
    JSON marking the error so the sidecar always exists; returns False.
    """
    try:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)
        return True
    except Exception as exc:  # noqa: BLE001 -- staging must not fail on a sidecar
        print(f"WARNING: failed to write {path}: {exc}", flush=True)
        try:
            with open(path, "w", encoding="utf-8") as f:
                f.write('{"error": "metadata sidecar generation failed"}')
        except Exception as exc2:  # noqa: BLE001
            print(f"WARNING: could not write fallback {path}: {exc2}", flush=True)
        return False


def _build_sidecars(dest_h5, granule):
    """Return (dataset_dict, met_dict) read from the product HDF5.

    Reads only the /science/LSAR/identification group. Any missing field is
    omitted. Raises only on unexpected errors; the caller guards against that.
    """
    ident = "/science/LSAR/identification"
    with h5py.File(dest_h5, "r") as h5:
        start_time = _h5_str(h5, f"{ident}/zeroDopplerStartTime")
        end_time = _h5_str(h5, f"{ident}/zeroDopplerEndTime")
        bbox_geojson = _bounding_polygon_to_geojson(_h5_str(h5, f"{ident}/boundingPolygon"))

        met = {
            "GranuleName": granule,
            "product_type": _h5_str(h5, f"{ident}/productType"),
            "product_level": _h5_str(h5, f"{ident}/productLevel"),
            "Product_Version": _h5_str(h5, f"{ident}/productVersion"),
            "Product_DOI": _h5_str(h5, f"{ident}/productDoi"),
            "Polarization": _h5_str(h5, f"{ident}/listOfFrequencies"),
            "track_number": _h5_num(h5, f"{ident}/trackNumber"),
            "frame_number": _h5_num(h5, f"{ident}/frameNumber"),
            "absolute_orbit_number": _h5_num(h5, f"{ident}/absoluteOrbitNumber"),
            "Direction": _h5_str(h5, f"{ident}/orbitPassDirection"),
            "look_direction": _h5_str(h5, f"{ident}/lookDirection"),
            "mission_id": _h5_str(h5, f"{ident}/missionId"),
            "radar_band": _h5_str(h5, f"{ident}/radarBand"),
            "processing_type": _h5_str(h5, f"{ident}/processingType"),
            "processing_center": _h5_str(h5, f"{ident}/processingCenter"),
            "Production_DateTime": _h5_str(h5, f"{ident}/processingDateTime"),
            "composite_release_id": _h5_str(h5, f"{ident}/compositeReleaseId"),
            "RefRadarStartDateTime": start_time,
            "RefRadarStopDateTime": end_time,
            "Bounding_Polygon": _h5_str(h5, f"{ident}/boundingPolygon"),
        }
    met = {k: v for k, v in met.items() if v is not None}

    dataset = {"version": met.get("Product_Version") or "v1.0"}
    if bbox_geojson is not None:
        dataset["location"] = {"type": "polygon", "coordinates": bbox_geojson}
    if start_time is not None:
        dataset["starttime"] = start_time
    if end_time is not None:
        dataset["endtime"] = end_time
    return dataset, met


stage_root = os.path.dirname(os.path.abspath(output_dir))  # parallel to output/

staged = []
for src in deliverables:
    if not os.path.exists(src):
        raise FileNotFoundError(f"expected SAS output not found: {src}")

    with h5py.File(src, "r") as h5:
        granule_id = _h5_str(h5, "/science/LSAR/identification/granuleId")
    if granule_id is None:
        raise ValueError(f"no granuleId in {src}")
    if "{" in granule_id or "}" in granule_id:
        raise ValueError(f"granuleId still has unfilled tokens: {granule_id!r} (from {src})")

    granule = granule_id[:-3] if granule_id.endswith(".h5") else granule_id
    dest_dir = os.path.join(stage_root, granule)
    os.makedirs(dest_dir, exist_ok=True)

    dest_h5 = os.path.join(dest_dir, granule + ".h5")
    shutil.move(src, dest_h5)
    shutil.copy2(local_rc, os.path.join(dest_dir, f"{granule}.rc.yaml"))

    # Build + write the sidecars defensively: sidecar generation must NEVER fail
    # staging. If reading metadata from the HDF5 raises for any reason, fall back
    # to static error payloads so both files still exist (a downstream consumer
    # then sees the error marker rather than a silently missing sidecar).
    dataset_path = os.path.join(dest_dir, granule + ".dataset.json")
    met_path = os.path.join(dest_dir, granule + ".met.json")
    try:
        dataset, met = _build_sidecars(dest_h5, granule)
    except Exception as exc:  # noqa: BLE001 -- never let staging die on metadata
        print(f"WARNING: metadata extraction failed for {dest_h5}: {exc}", flush=True)
        dataset = {"version": "v1.0", "error": "metadata extraction failed"}
        met = {"GranuleName": granule, "error": "metadata extraction failed"}
    _safe_write_json(dataset_path, dataset)
    _safe_write_json(met_path, met)

    staged.append(dest_dir)
    print(f"staged {src} -> {dest_h5} (+ .dataset.json, .met.json)")

print(f"\n{len(staged)} product(s) staged:")
for d in staged:
    print(" ", d)